# Digital PID Control - Part 1

![](./images/p19.png)

## Required imports

In [2]:
from IPython.core.display import HTML
import numpy as np
from sympy import *
from mathprint import *

## Define symbols

In [3]:
P, I, D   = symbols('P I D', positive=True)

Ts, N     = symbols('T_s N', positive=True)
s, sd, si = symbols('s s_D s_I', complex=True)
z         = symbols('z')

e, u      = symbols('e u')
e0, e1, e2, u0, u1, u2 = symbols('e_k e_{k-1} e_{k-2} u_k u_{k-1} u_{k-2}')

## Continuous-time PID with derivative filter

Let us start with defining our PID control in a continuous form. Here, we are going to follow **MATLAB Simulink** implementation. In Simulink, the derivative term is a first-order low-pass filter.

$$ U(s) = \Bigg( P + I \underbrace{\boxed{\frac{1}{s}}}_{\text{integrator}} + D \frac{N}{1+N \underbrace{\boxed{\frac{1}{s}}}_{\text{filter}}} \Bigg) E(s) $$

where:
* $P$ is the proportional gain
* $I$ is the integral gain
* $D$ is the derivative gain
* $N$ is the filter coefficient
* $e(t)$ or $E(s)$ is the input to the controller, which is the error between the target and the actual output of the system
* $u(t)$ or $U(s)$ is the computed control signal that will be sent to the system

In [4]:
Gpid = P + I/s + D*N/(1+N/s)   # PID-control
mprint("G(s)= ", latex(Gpid))

<IPython.core.display.Math object>

As for the discretization methods, there are 3 available methods: tarpezoidal (bilinear transform), forward Euler, and backward Euler. Integrator and filter can use different discretization method.

From this point onwards, let us introduce $T_s$ as the sampling period of the discrete-time PID control.

## Discrete-time PID with derivative filter

###  Using forward Euler method

Let us use the forward Euler method for both the integral term and filter.

$$ \frac{1}{s} \longleftarrow \frac{T_s}{z-1} $$

In [5]:
Gpid = P + I/s + D*N/(1+N/s)   # PID-control
Gpid = Gpid.subs( 1/s, Ts/(z-1) )
mprint("G_{PID}=", latex(Gpid))

Gpid = factor(Gpid)
Gpid = collect(Gpid, [z, z**2, Ts])
mprintb("G_{PID}=", latex(Gpid))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

Now, let us apply the input and compute the control output.

In [6]:
eq = Eq(u, Gpid * e)
mprint(latex(eq))

eq = factor(eq)
mprint(latex(eq))

eq = Eq(numer(eq.rhs), eq.lhs * denom(eq.rhs))
mprint(latex(eq))

eq = expand(Eq(eq.lhs/z**2/Ts/N, eq.rhs/z**2/Ts/N ))
mprint(latex(eq))

eq = eq.subs(e/z**2, e2).subs(e/z, e1).subs(e, e0).subs(u/z**2, u2).subs(u/z, u1).subs(u, u0)
mprint("\\small ", latex(eq))

eq = Eq(collect(eq.rhs, [u0, u1, u2]), collect(eq.lhs, [e0, e1, e2]) )
mprintb("\\small ", latex(eq))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

#### Comparison with Simulink

The following figure will be our general PID subsytem, where we can implement the discrete PID algorithm inside the MATLAB function block.

![](./images/p14.png)

The MATLAB function:
```
function u = PID(u1, u2, e0, e1, e2)
% Integral term     : Forward Euler:
% Derivative filter : Forward Euler

P = 0.12;
D = 0.1;
I = 1.2;
N = 100;
Ts = 1e-3;
NTs = N * Ts;

E0 = D/Ts + P/NTs;
E1 = -2*D/Ts + I/N + P -2*P/NTs;
E2 = D/Ts + I*Ts -I/N -P + P/NTs;

rhs = (1 - 2/NTs) * u1 + (-1 + 1/NTs)*u2;
lhs = E0 * e0 + E1 * e1 + E2 * e2;
u = (lhs - rhs) * NTs;
```

To compare our PID algorithm with Simulink, we then use the following scenario.

![](./images/p15.png)

![](./images/p16.png)

### Using bilinear transform method

Let us use the forward Euler method for both the integral term and filter.

$$ \frac{1}{s} \longleftarrow \frac{T_s}{2} \frac{z+1}{z-1} $$

In [7]:
Gpid = P + I/s + D*N/(1+N/s)     # PID-control
Gpid = Gpid.subs(1/s, Ts/2 * (z+1)/(z-1) )
mprint("G_{PID}=", latex(Gpid))

Gpid = factor(Gpid)

Gpid = collect(numer(Gpid), [z**2, z]) /  simplify(denom(Gpid))
mprintb("G_{PID}=", latex(Gpid))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

Now, let us apply the input to the output.

In [8]:
eq = Eq(u, Gpid * e)
mprint("\\small ", latex(eq))

eq = Eq(numer(eq.rhs), eq.lhs * denom(eq.rhs))
mprint("\\small ", latex(eq))

eq = expand(Eq(eq.lhs/z**2/Ts/N/8, eq.rhs/z**2/Ts/N/8 ))
mprint("\\small ", latex(eq))

eq = eq.subs(e/z**2, e2).subs(e/z, e1).subs(e, e0).subs(u/z**2, u2).subs(u/z, u1).subs(u, u0)
mprint("\\small ", latex(eq))

eq_ = Eq(collect(eq.rhs, [u0, u1, u2]), collect(expand(eq.lhs), [e0, e1, e2]))
mprintb("\\small ", latex(eq_))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

#### Comparison with Simulink


The Simulink subsystem and testing scenario for the bilinear PID is similar to the forward-Euler PID from the previous section. The only difference is in the MATLAB function.

The MATLAB function:

```
function u = PID(u1, u2, e0, e1, e2)
% Integral term     : bilinear
% Derivative filter : bilinear

P = 0.12;
D = 0.1;
I = 1.2;
N = 100;
Ts = 1e-3;
NTs = N * Ts;

E0 = D/(2*Ts) + I*Ts/8 + I/(4*N) + P/4 + P/(2*NTs);
E1 = -D/Ts + I*Ts/4 - P/NTs;
E2 = D/(2*Ts) + I*Ts/8 - I/(4*N) - P/4 + P/(2*NTs);

rhs = (-1/NTs) * u1 + (-1/4 + 1/(2*NTs))*u2;
lhs = E0 * e0 + E1 * e1 + E2 * e2;
u = (lhs - rhs) / (1/4 + 1/(2*NTs));

```


## Downloads

* [Simulink file for forward-Euler PID](https://www.dropbox.com/scl/fi/e82galim0xalsd4ya2bg9/mypid1.slx?rlkey=f1gv08mb8m8fci54xqltybwll&dl=0)
* [Simulink file for bilinear PID](https://www.dropbox.com/scl/fi/469bgqh589eacz7lbrthq/mypid2.slx?rlkey=1gsb4naa980vuuohs0og4l9sx&dl=0)

MATLAB version: R2024b
